# 07 — Small training example with smolVLA

**Objective:** check the training pipeline with the VLA.
**Revision: recorded-start-eval-v1.** Run top to bottom for a fresh training smoke test. To evaluate an existing checkpoint, run setup/data/adapter cells, set `checkpoint_dir` to that saved folder, then run the checkpoint-loading and evaluation cells; skip base-model initialization and training. Evaluation always restores the recorded training scene. It does not resume a previous rollout.

Continuing training after loading a checkpoint rebuilds the optimizer for that model (fresh optimizer state). The 500-step evaluation is a behavior diagnostic, shorter than the 4,466-step demonstration. No successful control is guaranteed by this notebook.


In [ ]:
!git clone https://github.com/huggingface/lerobot.git
%cd lerobot
!pip install -e ".[libero]"
!pip install -e ".[smolvla]"

In [ ]:
!pip install num2words

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import zipfile
import os

project = Path("/content/drive/MyDrive/Projects/humanoid-challenge")
SOURCE_RUN = "20261013T091720_023364Z"
archive_path = project / "data/lerobot_exports" / SOURCE_RUN / "lerobot_dataset.zip"
dataset_path = Path("/content/humanoid_dataset_exports") / SOURCE_RUN / "lerobot_dataset"

assert archive_path.is_file(), f"Archive missing: {archive_path}"

if not (dataset_path / "meta/info.json").is_file():
    dataset_path.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive_path, "r") as archive:
        archive.extractall(dataset_path)

assert (dataset_path / "meta/info.json").is_file()
assert (dataset_path / "data").is_dir()

print("Dataset ready:", dataset_path)

In [ ]:
import h5py
from torch.utils.data import DataLoader
from lerobot.datasets.lerobot_dataset import LeRobotDataset

REPO_ID = "local/humanoid_pilot01_smoke"
train_set = LeRobotDataset(repo_id=REPO_ID, root=dataset_path)
print("Frames:", len(train_set))
print("Episodes:", train_set.meta.total_episodes)
print("FPS:", train_set.meta.fps)

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

from libero.libero import benchmark
suite = benchmark.get_benchmark_dict()['libero_spatial']()
task_name = "pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate"

task_id = next(i for i in range(suite.n_tasks) if suite.get_task(i).name == task_name)
task = suite.get_task(task_id)

task_id, task.language

In [ ]:
import numpy as np
import torch
from scipy.spatial.transform import Rotation

def make_policy_observation(obs, instruction):
    # Quaternion from LIBERO uses XYZW ordering.
    rotation_vector = Rotation.from_quat(obs["robot0_eef_quat"]).as_rotvec()

    state = np.concatenate([
        obs["robot0_eef_pos"],
        rotation_vector,
        obs["robot0_gripper_qpos"],
    ]).astype(np.float32)

    def prepare_image(raw_image):
        # Match notebook 06: upright RGB.
        image = np.ascontiguousarray(raw_image[::-1])

        # HWC uint8 → CHW float32 in [0, 1].
        return torch.from_numpy(image).permute(2, 0, 1).float()/ 255.0


    return {
        "observation.images.image": prepare_image(obs["agentview_image"]),
        "observation.images.image2": prepare_image(obs["robot0_eye_in_hand_image"]),
        "observation.state": torch.from_numpy(state),
        "task": instruction,
    }


In [ ]:
import gc
import torch

from lerobot.configs.types import FeatureType, PolicyFeature
from lerobot.policies.smolvla.configuration_smolvla import SmolVLAConfig
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors

assert torch.cuda.is_available(), "Enable a GPU runtime in Colab first."

# Release the previous model before loading another copy.
if "policy" in globals():
    del policy

gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "lerobot/smolvla_base"

config = SmolVLAConfig.from_pretrained(MODEL_ID)
config.device = "cuda"

config.input_features = {
    "observation.images.image": PolicyFeature(
        type=FeatureType.VISUAL,
        shape=(3, 256, 256),
    ),
    "observation.images.image2": PolicyFeature(
        type=FeatureType.VISUAL,
        shape=(3, 256, 256),
    ),
    "observation.state": PolicyFeature(
        type=FeatureType.STATE,
        shape=(8,),
    ),
}

config.output_features = {
    "action": PolicyFeature(
        type=FeatureType.ACTION,
        shape=(7,),
    ),
}

config.chunk_size = 50
config.n_action_steps = 10
config.empty_cameras = 0

# Preserve the checkpoint's internal padded dimensions.
assert config.max_state_dim >= 8
assert config.max_action_dim >= 7

policy = SmolVLAPolicy.from_pretrained(
    MODEL_ID,
    config=config,
    strict=True,
).to("cuda")

print("Inputs:", policy.config.input_features)
print("Outputs:", policy.config.output_features)

In [ ]:
preprocess, postprocess = make_pre_post_processors(
    policy.config,
    pretrained_path=None,
    dataset_stats=train_set.meta.stats,
)

In [ ]:
from torch.utils.data import DataLoader
from lerobot.datasets.lerobot_dataset import LeRobotDataset

training_dataset = LeRobotDataset(
    repo_id=REPO_ID,
    root=dataset_path,
    delta_timestamps={
        "action": [
            i / train_set.meta.fps
            for i in range(policy.config.chunk_size)
        ]
    },
)

loader = DataLoader(
    training_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
)

raw_batch = next(iter(loader))
model_batch = preprocess(raw_batch)

for key in [
    "observation.images.image",
    "observation.images.image2",
    "observation.state",
    "action",
]:
    value = model_batch[key]
    print(key, tuple(value.shape), value.device)
    assert torch.isfinite(value).all()

assert model_batch["observation.state"].shape == (1, 8)
assert model_batch["action"].shape == (1, 50, 7)

In [ ]:
def ensure_training_optimizer():
    """Reuse Adam state only while it belongs to the current policy instance."""
    global optimizer, trainable_parameters
    trainable_parameters = [p for p in policy.parameters() if p.requires_grad]
    current_ids = {id(p) for p in trainable_parameters}
    previous = globals().get("optimizer")
    optimizer_ids = (
        {id(p) for group in previous.param_groups for p in group["params"]}
        if previous is not None else set()
    )
    if optimizer_ids != current_ids:
        optimizer = torch.optim.AdamW(trainable_parameters, lr=1e-5, weight_decay=1e-2)
        print("Created optimizer for current policy (fresh optimizer state).")
    assert current_ids == {id(p) for group in optimizer.param_groups for p in group["params"]}



In [ ]:
policy.train()
policy.reset()

ensure_training_optimizer()
print("Trainable parameters:", sum(p.numel() for p in trainable_parameters))

raw_batch = next(iter(loader))
model_batch = preprocess(raw_batch)

optimizer.zero_grad(set_to_none=True)
loss, metrics = policy.forward(model_batch)

assert torch.isfinite(loss).item(), "Loss is NaN or infinite."
loss.backward()
gradient_norm = torch.nn.utils.clip_grad_norm_(trainable_parameters, max_norm=1.0, error_if_nonfinite=True)
optimizer.step()

print("Training loss:", loss.detach().item())
print("Gradient norm before clipping:", float(gradient_norm))
print("One optimizer step completed.")

In [ ]:
import time

ensure_training_optimizer()
policy.train()
iterator = iter(loader)

loss_history = []
started = time.monotonic()
n_step = 2000

# Before the training loop:
training_details = []

for step_index in range(n_step):
    try:
        raw_batch = next(iterator)
    except StopIteration:
        iterator = iter(loader)
        raw_batch = next(iterator)

    model_batch = preprocess(raw_batch)

    optimizer.zero_grad(set_to_none=True)

    loss, metrics = policy.forward(model_batch)
    assert torch.isfinite(loss).item(), "Non-finite loss."

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        trainable_parameters,
        max_norm=1.0,
        error_if_nonfinite=True,
    )

    optimizer.step()
    loss_history.append(float(loss.detach().cpu()))

    # Inside it, after optimizer.step():
    training_details.append({
        "step": step_index + 1,
        "frame_indices": raw_batch["index"].detach().cpu().tolist(),
        "loss": float(loss.detach().cpu()),
        "gradient_norm_before_clipping": float(grad_norm),
    })

    if (step_index + 1) % 20 == 0:
        recent_mean = sum(loss_history[-20:]) / 20

        print(
            f"Step {step_index + 1}/{n_step} | "
            f"mean loss: {recent_mean:.4f} | "
            f"elapsed: {time.monotonic() - started:.0f}s",
            flush=True,
        )

In [ ]:
import json
from datetime import datetime, timezone

checkpoint_dir = project / "outputs/smolvla_smoke" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
checkpoint_dir.mkdir(parents=True, exist_ok=False)

policy.save_pretrained(checkpoint_dir)

preprocess.save_pretrained(checkpoint_dir, config_filename="policy_preprocessor.json")
postprocess.save_pretrained(checkpoint_dir, config_filename="policy_postprocessor.json")

(checkpoint_dir / "training_log.json").write_text(
    json.dumps({
        "additional_steps": len(loss_history),
        "batch_size": loader.batch_size,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "loss_history": loss_history,
        "dataset_root": str(dataset_path),
        "purpose": "single-episode training smoke test",
    }, indent=2)
)

(checkpoint_dir / "training_details.json").write_text(
    json.dumps(training_details, indent=2)
)

print("Saved:", checkpoint_dir)

In [ ]:
import gc
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors

# Drop optimizer references to the previous model before evaluation reload.
optimizer = None
trainable_parameters = []
if "policy" in globals():
    del policy
gc.collect()
torch.cuda.empty_cache()

policy = SmolVLAPolicy.from_pretrained(checkpoint_dir).to("cuda").eval()

preprocess, postprocess = make_pre_post_processors(
    policy.config,
    pretrained_path=checkpoint_dir,
    preprocessor_overrides={
        "device_processor": {"device": "cuda"},
    },
)

print(policy.config.output_features)
# Expect action shape=(7,)

In [ ]:
from libero.libero import get_libero_path
from libero.libero.envs import OffScreenRenderEnv

# retrieve a specific task
bddl_path = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)

# Close an older environment before constructing a new one.
if "env" in globals():
    env.close()

# Evaluation environment
env = OffScreenRenderEnv(
    bddl_file_name=bddl_path,
    camera_heights=256,
    camera_widths=256,
    horizon=5000,
    ignore_done=True,
)

In [ ]:
import h5py

source_episode = (
    project / "data/robot/pilot_01"
    / "20261012T161104_719335Z" / "episode.h5"
)

def reset_to_recorded_start():
    """Restore scene XML and state; check the pre-action state matches frame zero."""
    with h5py.File(source_episode, "r") as f:
        initial_state = f["initial_sim_state"][:]
        model_xml = f["model_xml"][()]
        first_state = f["state"][0]
    if isinstance(model_xml, bytes):
        model_xml = model_xml.decode()
    env.reset()
    env.reset_from_xml_string(model_xml)
    restored_obs = env.set_init_state(initial_state)
    np.testing.assert_allclose(
        np.asarray(env.get_sim_state()), initial_state, atol=1e-6, rtol=0,
        err_msg="Simulator did not restore the recorded state",
    )
    live_state = np.r_[restored_obs["robot0_eef_pos"],
                       restored_obs["robot0_eef_quat"],
                       restored_obs["robot0_gripper_qpos"]]
    np.testing.assert_allclose(live_state, first_state, atol=1e-5, rtol=0)
    policy.reset()
    preprocess.reset()
    postprocess.reset()
    print("Evaluation reset: recorded training scene; state checks passed.")
    return restored_obs

obs = reset_to_recorded_start()


In [ ]:
with torch.inference_mode():
    policy_observation = make_policy_observation(obs, task.language)
    model_batch = preprocess(policy_observation)

    predicted_action = policy.select_action(model_batch)
    action_tensor = postprocess(predicted_action)

action = action_tensor.squeeze(0).detach().cpu().numpy()

print("Action shape:", action.shape)
print("Action:", action)

assert action.shape == (7,)
assert np.isfinite(action).all()

# Clear the queued actions from this diagnostic prediction.
policy.reset()
preprocess.reset()
postprocess.reset()

In [ ]:
import imageio.v2 as imageio
from IPython.display import Video, display
from datetime import datetime, timezone

obs = reset_to_recorded_start()

policy.eval()
policy.reset()
preprocess.reset()
postprocess.reset()

frames = []
rollout_log = []
initial_eef_position = obs["robot0_eef_pos"].copy()
success = False
clipped_steps = 0
max_steps = 500
stop_reason = "step limit"

low, high = env.env.action_spec

with torch.inference_mode():
    for step_index in range(max_steps):
        policy_observation = make_policy_observation(obs, task.language)
        model_batch = preprocess(policy_observation)

        action_tensor = postprocess(policy.select_action(model_batch))
        action = action_tensor.squeeze(0).detach().cpu().numpy()

        assert action.shape == (7,)
        assert np.isfinite(action).all()

        bounded_action = np.clip(action, low, high)
        clipped_steps += int(np.any(bounded_action != action))

        previous_eef = obs["robot0_eef_pos"].copy()
        obs, reward, done, info = env.step(bounded_action)
        rollout_log.append({
            "step": step_index,
            "eef_before": previous_eef.tolist(),
            "eef_after": obs["robot0_eef_pos"].tolist(),
            "predicted_action": action.tolist(),
            "executed_action": bounded_action.tolist(),
            "success": bool(env.check_success()),
        })

        frames.append(
            np.concatenate([
                obs["agentview_image"][::-1],
                obs["robot0_eye_in_hand_image"][::-1],
            ], axis=1).copy()
        )

        success = bool(env.check_success())

        if (step_index + 1) % 50 == 0:
            print(f"Step {step_index + 1}/{max_steps}", flush=True)

        if success or done:
            stop_reason = "task success" if success else "environment ended"
            break

eval_dir = project / "outputs/smolvla_eval" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
eval_dir.mkdir(parents=True, exist_ok=False)

video_path = eval_dir / "rollout.mp4"
imageio.mimsave(str(video_path), frames, fps=20)

print("Steps:", len(frames))
print("Task success:", success)
print("Stop reason:", stop_reason)
print("Steps requiring action clipping:", clipped_steps)

import json
(eval_dir / "evaluation_report.json").write_text(json.dumps({
    "revision": "recorded-start-eval-v1",
    "checkpoint": str(checkpoint_dir),
    "source_episode": str(source_episode),
    "reset_mode": "recorded model XML and initial simulator state",
    "steps": len(frames),
    "task_success": success,
    "stop_reason": stop_reason,
    "clipped_steps": clipped_steps,
    "eef_displacement_m": (obs["robot0_eef_pos"] - initial_eef_position).tolist(),
    "release_verified": False,
}, indent=2))
(eval_dir / "actions.json").write_text(json.dumps(rollout_log))
print("Evaluation artifacts:", eval_dir)
display(Video(str(video_path), embed=True))

The rollout above restores the recorded start internally. Do not restore the state afterward and assume it changes the video already produced. The saved report and action log identify the evaluated checkpoint and scene.


In [ ]:
window = 20
smoothed_loss = np.convolve(
    loss_history,
    np.ones(window) / window,
    mode="valid",
)

import matplotlib.pyplot as plt

plt.plot(smoothed_loss)
plt.xlabel("Training step, offset by 19")
plt.ylabel("Mean training loss over 20 steps")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
policy.eval()

# Sample observations across the first half of the demonstration.
indices = [0, 500, 1000, 1700, 1800, 1900, 2000, 2100, 2300]
results = []

for index in indices:
    sample = training_dataset[index]

    # Use the current observation and instruction, not target actions.
    observation = {
        key: value
        for key, value in sample.items()
        if key.startswith("observation.") or key == "task"
    }

    predicted_actions = []

    # Repeat because action generation is stochastic.
    for repeat in range(3):
        torch.manual_seed(100 + repeat)
        policy.reset()
        preprocess.reset()
        postprocess.reset()

        with torch.inference_mode():
            model_batch = preprocess(dict(observation))
            prediction = postprocess(policy.select_action(model_batch))

        predicted_actions.append(prediction.squeeze(0).detach().cpu().numpy())

    # training_dataset contains action chunks; compare the first action.
    target = sample["action"][0].cpu().numpy()
    predictions = np.stack(predicted_actions)
    print(
        f"Frame {index} | "
        f"target grip: {target[6]:+.2f} | "
        f"predicted grips: {np.round(predictions[:, 6], 3)}"
    )

    results.append({
        "frame": index,
        "target": target,
        "prediction_mean": predictions.mean(axis=0),
        "translation_mae": np.abs(predictions[:, :3] - target[:3]).mean(),
        "rotation_mae": np.abs(predictions[:, 3:6] - target[3:6]).mean(),
        "gripper_mae": np.abs(predictions[:, 6] - target[6]).mean(),
    })

for result in results:
    print(f"\nFrame {result['frame']}")
    print("Target:    ", np.round(result["target"], 4))
    print("Prediction:", np.round(result["prediction_mean"], 4))
    print(
        "MAE — translation / rotation / gripper:",
        round(float(result["translation_mae"]), 4),
        round(float(result["rotation_mae"]), 4),
        round(float(result["gripper_mae"]), 4),
    )

## Diagnose high-loss frames

Run the cells below after training, without rerunning training or the simulator. They align each batch-size-one loss with its source phase and future action chunk. Losses were measured at different optimizer steps with stochastic flow-matching noise; these plots show associations, not controlled per-frame difficulty. Normalization statistics are inspected, not modified.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import json
import numpy as np

# Recover persisted details if the runtime was restarted.
if "training_details" not in globals() or not training_details:
    training_details = json.loads((checkpoint_dir / "training_details.json").read_text())
details = pd.DataFrame(training_details)
assert len(details) > 0
assert details["frame_indices"].map(len).eq(1).all(), (
    "This per-frame analysis assumes batch size 1; batch losses cannot be assigned to individual frames otherwise."
)
# Verify that details belong to the selected saved checkpoint's training run.
saved_log = json.loads((checkpoint_dir / "training_log.json").read_text())
assert len(details) == len(saved_log["loss_history"])
np.testing.assert_allclose(details["loss"], saved_log["loss_history"], rtol=1e-6)
details["frame"] = details["frame_indices"].map(lambda indices: int(indices[0]))
source_episode = project / "data/robot/pilot_01/20261012T161104_719335Z/episode.h5"
with h5py.File(source_episode, "r") as f:
    phases = f["phase"].asstr()[:]
    reference_actions = f["action"][:]
assert len(reference_actions) == len(training_dataset)
assert details["frame"].between(0, len(phases)-1).all()
details["phase"] = phases[details["frame"].to_numpy()]
details["phase_group"] = details["phase"].str.replace(r"human_transfer_\d+", "human_transfer", regex=True)

def to_numpy(value):
    return value.detach().cpu().numpy() if hasattr(value, "detach") else np.asarray(value)

action_stats = train_set.meta.stats["action"]
action_mean = to_numpy(action_stats["mean"]).reshape(7)
action_std = to_numpy(action_stats["std"]).reshape(7)
# Match MEAN_STD's epsilon convention for a diagnostic z-score.
normalized_actions = (reference_actions - action_mean) / (action_std + 1e-8)
chunk_length = policy.config.chunk_size
chunk_peak = np.array([
    np.abs(normalized_actions[i:i+chunk_length]).max()
    for i in range(len(reference_actions))
])
details["chunk_max_abs_z"] = chunk_peak[details["frame"].to_numpy()]
details["clipped_gradient"] = details["gradient_norm_before_clipping"] > 1.0
print("Logged steps:", len(details))
print("Fraction of steps gradient-clipped:", details["clipped_gradient"].mean())
display(details.nlargest(15, "loss")[["step", "frame", "phase", "loss", "gradient_norm_before_clipping", "chunk_max_abs_z"]])
display(details.groupby("phase_group").agg(
    samples=("loss", "size"), median_loss=("loss", "median"),
    mean_loss=("loss", "mean"), max_loss=("loss", "max")
).sort_values("mean_loss", ascending=False))

# Persist the full log rather than a truncated notebook display.
(checkpoint_dir / "training_details.json").write_text(json.dumps(training_details, indent=2))
details.to_csv(checkpoint_dir / "training_diagnostics.csv", index=False)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes[0, 0].plot(details["step"], details["loss"], alpha=.3, label="Individual")
axes[0, 0].plot(details["step"], details["loss"].rolling(50, min_periods=1).median(), label="50-step median")
axes[0, 0].set(xlabel="Optimizer step", ylabel="Loss", yscale="log")
axes[0, 0].legend()
points = axes[0, 1].scatter(details["frame"], details["loss"], c=details["step"], s=10, alpha=.7)
axes[0, 1].set(xlabel="Demonstration frame", ylabel="Loss", yscale="log")
fig.colorbar(points, ax=axes[0, 1], label="Optimizer step")
axes[1, 0].scatter(details["chunk_max_abs_z"], details["loss"], s=10, alpha=.5)
axes[1, 0].set(xlabel="Largest |normalized target| in 50-action chunk", ylabel="Loss", yscale="log")
axes[1, 1].scatter(details["loss"], details["gradient_norm_before_clipping"], s=10, alpha=.5)
axes[1, 1].set(xlabel="Loss", ylabel="Gradient norm before clipping", xscale="log", yscale="log")
for ax in axes.flat:
    ax.grid(alpha=.2)
plt.tight_layout()
fig.savefig(checkpoint_dir / "training_diagnostics.png", dpi=150)
plt.show()

# Inspect the initial settling/orientation segment and its action scale.
labels = ["dx", "dy", "dz", "rx", "ry", "rz", "gripper"]
end = min(350, len(reference_actions))
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for d, label in enumerate(labels):
    axes[0].plot(reference_actions[:end, d], label=label)
    axes[1].plot(normalized_actions[:end, d], label=label)
axes[0].set_ylabel("Raw controller command")
axes[1].set(xlabel="Demonstration frame", ylabel="Normalized command")
axes[0].legend(ncol=7)
for ax in axes:
    ax.grid(alpha=.2)
plt.tight_layout()
fig.savefig(checkpoint_dir / "initial_action_scales.png", dpi=150)
plt.show()


In [ ]:
# Preview unique high-loss observations and their FUTURE target sequences.
# A loss at frame i supervises i ... i+49, not just action i.
worst = details.sort_values("loss", ascending=False).drop_duplicates("frame").head(4)
fig, axes = plt.subplots(len(worst), 3, figsize=(14, 3.5*len(worst)), squeeze=False)
with h5py.File(source_episode, "r") as f:
    for row, (_, entry) in enumerate(worst.iterrows()):
        index = int(entry["frame"])
        for col, camera in enumerate(["agentview_image", "wrist_image"]):
            axes[row, col].imshow(f[camera][index])
            axes[row, col].axis("off")
            axes[row, col].set_title(f"Frame {index}: {entry['phase']} | {camera}")
        stop = min(index+chunk_length, len(reference_actions))
        for d, label in enumerate(labels):
            axes[row, 2].plot(np.arange(index, stop), normalized_actions[index:stop, d], label=label)
        axes[row, 2].set_title(f"Target chunk | historical loss {entry['loss']:.2f}")
        axes[row, 2].set(xlabel="Frame", ylabel="Normalized action")
        axes[row, 2].grid(alpha=.2)
axes[0, 2].legend(ncol=4, fontsize=8)
plt.tight_layout()
fig.savefig(checkpoint_dir / "high_loss_observations.png", dpi=150)
plt.show()
